# Memory viewer

Every flagged memory in NAMS, numbered from 0, oldest first. Pick one. The before half is the reply that acted: its question, its words, the picture it was shown, and that picture's Settings. The after half is the reply that wrote the flag, with the picture that reply saw.

Tokens are clickable when a gaze trace was stored with that reply. A click prints **s** and **v** and draws them on the picture that reply was shown. Blue is s, red is v. Click the same token again to clear the dots. The other picture is left alone.

No model is loaded. Neo4j has to be up (`bash scripts/vast_neo4j_launch.sh`). **Refresh** reads NAMS again.


In [ ]:
import asyncio
import html
import io
import json
import os
import sys
import threading
import traceback
from pathlib import Path

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import ipywidgets as widgets
from IPython.display import display
from PIL import Image, ImageDraw

from agent.game_io import fmt3, settings_json_with_openings
from agent.image_store import load_image
from agent import memory as mem
from neural_net.gaze_replay import valid_trace

PX = 420
S_COLOR = (30, 90, 255)
V_COLOR = (255, 0, 0)

_loop = asyncio.new_event_loop()
_thread = threading.Thread(target=_loop.run_forever, name="memory-viewer", daemon=True)
_thread.start()


def _run(coro):
    return asyncio.run_coroutine_threadsafe(coro, _loop).result()


def _gaze(raw):
    if not raw:
        return None
    meta = raw
    if isinstance(raw, str):
        try:
            meta = json.loads(raw)
        except ValueError:
            return None
    if not isinstance(meta, dict):
        return None
    gaze = meta.get("gaze")
    return gaze if isinstance(gaze, dict) else None


async def _fetch(client):
    flags = await mem.load_all_flagged(client)
    rows = await client.query.cypher(
        "MATCH (m:Message) WHERE m.metadata CONTAINS '\"failure\"' "
        "RETURN m.id AS id, m.metadata AS metadata"
    )
    after = {}
    for row in rows or []:
        item = dict(row)
        after[str(item.get("id") or "")] = _gaze(item.get("metadata"))
    return flags, after


async def _connect():
    client = await mem.connect()
    flags, after = await _fetch(client)
    return client, flags, after


client, flags, after_gaze = _run(_connect())
state = {
    "client": client,
    "flags": flags,
    "after_gaze": after_gaze,
    "n": None,
    "selected": None,
    "gen": 0,
    "ignore": 0,
    "images": {},
    "tokens": {"before": None, "after": None},
}


def _xy3(pair):
    shown = fmt3(pair)
    if isinstance(shown, list) and len(shown) == 2:
        return "(" + str(shown[0]) + ", " + str(shown[1]) + ")"
    return str(shown)


def _settings_text(side):
    if not isinstance(side, dict):
        return ""
    if "error" in side:
        raw = side.get("settings")
        return raw if isinstance(raw, str) else ""
    raw = side.get("settings")
    if not isinstance(raw, str) or not raw:
        return ""
    if raw.startswith("[ERROR"):
        return raw
    opened = settings_json_with_openings(raw) or raw
    return json.dumps(fmt3(json.loads(opened)), indent=2)


def _png(image):
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return buf.getvalue()


def _dot(draw, size, x, y, color):
    width, height = size
    px = float(x) * (width - 1)
    py = (1.0 - float(y)) * (height - 1)
    r = 8
    draw.ellipse((px - r, py - r, px + r, py + r), fill=color, outline=(0, 0, 0))


def _open_png(path):
    file = Path(load_image(path))
    if not file.is_file():
        raise FileNotFoundError(f"flagged board image is missing: {file}")
    with Image.open(file) as im:
        return im.convert("RGB")


def _tokens(text, trace):
    if not isinstance(text, str) or not text:
        return None
    if not valid_trace(trace, text):
        return None
    rows = []
    for span, read in zip(trace["spans"], trace["reads"]):
        piece = text[int(span[0]):int(span[1])]
        rows.append((piece, read))
    return rows


def _visible(piece):
    if piece == "":
        return "\u2205"
    return piece.replace(" ", "\u00a0").replace("\n", "\u21b5")


def _token_html(side, rows):
    if rows is None:
        return ""
    gen = state["gen"]
    selected = state["selected"]
    parts = []
    for i, (piece, _read) in enumerate(rows):
        cls = "mv-tok"
        if selected == (side, i):
            cls += " mv-on"
        parts.append(
            f"<span class='{cls}' data-gen='{gen}' data-side='{side}' "
            f"data-i='{i}'>{html.escape(_visible(piece))}</span>"
        )
    return "<div class='mv-tokens'>" + "".join(parts) + "</div>"


def _plain(text):
    body = text if isinstance(text, str) else ""
    return "<pre class='mv-text'>" + html.escape(body) + "</pre>"


def _paint(side):
    image = state["images"].get(side)
    widget = before_img if side == "before" else after_img
    if image is None:
        widget.value = b""
        return
    selected = state["selected"]
    shown = image
    if selected is not None and selected[0] == side:
        rows = state["tokens"].get(side) or []
        _piece, read = rows[selected[1]]
        shown = image.copy()
        draw = ImageDraw.Draw(shown)
        _dot(draw, shown.size, read[0], read[1], S_COLOR)
        _dot(draw, shown.size, read[2], read[3], V_COLOR)
    widget.value = _png(shown)


def _side_block(item, side):
    if side == "before":
        question = item.get("before_question") or ""
        message = item.get("before_message") or ""
        board = item.get("before") or {}
        trace = item.get("gaze")
    else:
        question = item.get("after_question") or ""
        message = item.get("after_message") or item.get("content") or ""
        board = item.get("after") or {}
        trace = state["after_gaze"].get(str(item.get("id") or ""))
    rows = _tokens(message, trace)
    state["tokens"][side] = rows
    if isinstance(board, dict) and board.get("path") and "error" not in board:
        state["images"][side] = _open_png(board["path"])
        picture_note = ""
    elif isinstance(board, dict) and board.get("error"):
        state["images"][side] = None
        picture_note = "<pre>" + html.escape(str(board.get("error") or "")) + "</pre>"
    else:
        state["images"][side] = None
        picture_note = "<pre>(no picture)</pre>"
    words = _token_html(side, rows) if rows is not None else _plain(message)
    if rows is None and message:
        words += "<div class='mv-note'>No gaze trace for this reply.</div>"
    return (
        "<b>" + side + " question</b>" + _plain(question)
        + "<b>" + side + " message</b>" + words
        + picture_note
        + "<b>" + side + " Settings</b><pre class='mv-settings'>"
        + html.escape(_settings_text(board))
        + "</pre>"
    )


def _label(item):
    note = str(item.get("failure") or "").replace("\n", " ")[:80]
    sid = str(item.get("session_id") or "")[:8]
    return f"{item['n']}  {sid}  {note}"


def _options():
    if not state["flags"]:
        return [("(no flagged memories)", -1)]
    return [(_label(item), int(item["n"])) for item in state["flags"]]


def _show(n):
    state["gen"] += 1
    state["selected"] = None
    state["n"] = n
    state["images"] = {}
    state["tokens"] = {"before": None, "after": None}
    readout.value = "Click a token. Blue is s, red is v, on the picture that reply was shown."
    if n < 0 or n >= len(state["flags"]):
        before_text.value = ""
        after_text.value = ""
        before_img.value = b""
        after_img.value = b""
        head.value = f"<b>{len(state['flags'])} flagged memories.</b>"
        return
    item = state["flags"][n]
    sid = str(item.get("session_id") or "")
    head.value = (
        f"<b>Memory {n}</b> &nbsp; game {html.escape(sid[:8])}"
        f"<pre>{html.escape(str(item.get('failure') or ''))}</pre>"
    )
    before_text.value = _side_block(item, "before")
    after_text.value = _side_block(item, "after")
    _paint("before")
    _paint("after")


def on_memory(change):
    if change["name"] != "value":
        return
    _show(int(change["new"]))


def on_pick(change):
    raw = change["new"]
    if not raw or raw.count(",") != 3:
        return
    seq_s, gen_s, side, idx_s = raw.split(",")
    try:
        seq = int(seq_s)
        gen = int(gen_s)
        tok_i = int(idx_s)
    except ValueError:
        return
    if seq <= state["ignore"] or gen != state["gen"]:
        return
    if side not in ("before", "after"):
        return
    state["ignore"] = seq
    rows = state["tokens"].get(side)
    if not rows or tok_i < 0 or tok_i >= len(rows):
        return
    if state["selected"] == (side, tok_i):
        state["selected"] = None
        readout.value = "Click a token. Blue is s, red is v, on the picture that reply was shown."
    else:
        state["selected"] = (side, tok_i)
        piece, read = rows[tok_i]
        readout.value = (
            f"{side} token {tok_i} &nbsp; {html.escape(_visible(piece))}<br>"
            f"s={_xy3((read[0], read[1]))} &nbsp; v={_xy3((read[2], read[3]))}"
        )
        if state["images"].get(side) is None:
            readout.value += "<br>This reply has no picture to draw on."
    item = state["flags"][state["n"]]
    before_text.value = _side_block(item, "before")
    after_text.value = _side_block(item, "after")
    _paint("before")
    _paint("after")


def on_refresh(_):
    try:
        flags, after = _run(_fetch(state["client"]))
    except Exception:
        status.value = "Refresh failed."
        log.append_stdout(traceback.format_exc())
        return
    state["flags"] = flags
    state["after_gaze"] = after
    picker.unobserve(on_memory, names="value")
    opts = _options()
    picker.options = opts
    keep = state["n"] if state["n"] is not None else opts[0][1]
    values = [value for _label, value in opts]
    picker.value = keep if keep in values else values[0]
    picker.observe(on_memory, names="value")
    _show(int(picker.value))
    status.value = f"{len(flags)} flagged memories."


opts = _options()
picker = widgets.Dropdown(
    options=opts,
    value=opts[0][1],
    description="Memory:",
    layout=widgets.Layout(width="100%"),
)
refresh_btn = widgets.Button(description="Refresh")
status = widgets.HTML(f"<span>{len(flags)} flagged memories.</span>")
head = widgets.HTML()
readout = widgets.HTML("Click a token.")
legend = widgets.HTML(
    "<span style='color:#1e5aff'>\u25cf s</span> &nbsp; "
    "<span style='color:#ff0000'>\u25cf v</span>"
)
before_text = widgets.HTML()
after_text = widgets.HTML()
before_img = widgets.Image(format="png", width=PX)
after_img = widgets.Image(format="png", width=PX)
log = widgets.Output(layout=widgets.Layout(height="80px", overflow="auto"))
pick_box = widgets.Text(value="")
pick_box.add_class("mv-pick")
pick_box.layout.height = "0px"
pick_box.layout.visibility = "hidden"
pick_box.layout.overflow = "hidden"
picker.observe(on_memory, names="value")
pick_box.observe(on_pick, names="value")
refresh_btn.on_click(on_refresh)
_show(int(picker.value))

display(widgets.HTML(
    """
<style>
.mv-tokens {
  white-space: pre-wrap;
  line-height: 1.8;
  border: 1px solid #ccc;
  padding: 8px;
  max-height: 180px;
  overflow-y: auto;
  font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
}
.mv-tok { cursor: pointer; border-radius: 3px; }
.mv-tok:hover { background: #e8eefc; }
.mv-tok.mv-on { background: #ffe08a; }
.mv-text, .mv-settings {
  white-space: pre-wrap;
  max-height: 220px;
  overflow-y: auto;
  border: 1px solid #eee;
  padding: 6px;
}
.mv-note { color: #666; font-family: sans-serif; font-size: 12px; }
</style>
<script>
(function () {
  if (window.__mvTokenClick) { return; }
  window.__mvTokenClick = true;
  window.addEventListener("click", function (ev) {
    var span = ev.target && ev.target.closest && ev.target.closest(".mv-tok");
    if (!span) { return; }
    var box = document.querySelector(".mv-pick input");
    if (!box) { return; }
    window.__mvPickN = (window.__mvPickN || 0) + 1;
    var gen = span.getAttribute("data-gen") || "0";
    var side = span.getAttribute("data-side") || "";
    var i = span.getAttribute("data-i");
    box.value = window.__mvPickN + "," + gen + "," + side + "," + i;
    box.dispatchEvent(new Event("input", { bubbles: true }));
  }, true);
})();
</script>
"""
))
display(widgets.VBox([
    widgets.HBox([picker, refresh_btn]),
    status,
    head,
    readout,
    legend,
    widgets.HBox([
        widgets.VBox([
            widgets.HTML("<b>Before</b>"),
            before_text,
            before_img,
        ], layout=widgets.Layout(flex="1 1 0", min_width="440px")),
        widgets.VBox([
            widgets.HTML("<b>After</b>"),
            after_text,
            after_img,
        ], layout=widgets.Layout(flex="1 1 0", min_width="440px")),
    ], layout=widgets.Layout(align_items="flex-start")),
    pick_box,
    log,
]))
